In [5]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DATA_PATH = os.path.join("..", "out_tfg_reorders")
SUBSET_DATA = os.path.join(BASE_DATA_PATH, "subset")
ALL_REORDER_MODES = ["none", "polar", "cartesian"]
ALL_CLOUDS = ["bildstein_station1_xyz_intensity_rgb", "sg27_station8_intensity_rgb", "5145_54340", "Lille_0", "Paris_Luxembourg_6", "PNOA_2024_PNR_489-4672_NPC01"]

In [ ]:
def verificar_integridad_vecinos(data_path, clouds):
    """
    Recorre los directorios de resultados de los datasets indicados, abre cada CSV
    y comprueba que el 'avg_result_size' sea idéntico para los 3 modos de reordenación.
    """
    total_errores = 0
    total_archivos_revisados = 0
    
    print("Iniciando test de validación de resultados...")
    print("-" * 60)
    
    # 1. Iterar sobre los datasets proporcionados
    for dataset in clouds:
        # Construir la ruta al directorio del dataset específico
        dir_dataset = os.path.join(data_path, dataset)
        
        # Validar si el directorio realmente existe para evitar excepciones
        if not os.path.isdir(dir_dataset):
            print(f"⚠️ Aviso: El directorio '{dir_dataset}' no existe. Saltando...")
            continue
            
        # 2. Buscar todos los archivos .csv dentro de ese directorio
        # (Usa recursive=True por si acaso tus resultados están organizados en subcarpetas)
        patron_csv = os.path.join(dir_dataset, "**", "*.csv")
        archivos_csv = glob.glob(patron_csv, recursive=True)
        
        for ruta_csv in archivos_csv:
            total_archivos_revisados += 1
            nombre_archivo_corto = os.path.relpath(ruta_csv, data_path)
            
            try:
                # Leer el dataframe actual
                df = pd.read_csv(ruta_csv)
                
                # Comprobación de seguridad: verificar que las columnas necesarias existan
                if "reorder" not in df.columns or "avg_result_size" not in df.columns:
                    continue
                
                # Diccionario para almacenar el tamaño de vecinos de cada modo en este CSV
                valores_modos = {}
                
                # 3. Separar y extraer el valor medio de vecinos para cada uno de los 3 modos
                for modo in ALL_REORDER_MODES:
                    df_modo = df[df["reorder"] == modo]
                    
                    if df_modo.empty:
                        valores_modos[modo] = None
                    else:
                        # Extraemos el valor. Si hay varias filas por modo en el mismo CSV (por repeticiones),
                        # calculamos la media para asegurar un único valor comparable.
                        valores_modos[modo] = df_modo["avg_result_size"].mean()
                
                # 4. Verificar si falta algún modo en el CSV antes de comparar
                if any(v is None for v in valores_modos.values()):
                    # Si faltan datos de algún modo, no podemos comparar de forma justa
                    continue
                
                v_none = valores_modos['none']
                v_polar = valores_modos['polar']
                v_cartesian = valores_modos['cartesian']
                
                # 5. Comprobación matemática de discrepancias (tolerancia a fallos de coma flotante)
                # Compara si Polar o Cartesian difieren del modo base 'none'
                error_polar = not np.isclose(v_none, v_polar, rtol=1e-5, atol=1e-8)
                error_cartesian = not np.isclose(v_none, v_cartesian, rtol=1e-5, atol=1e-8)
                
                if error_polar or error_cartesian:
                    total_errores += 1
                    print(f"❌ DISCREPANCIA DETECTADA en: {nombre_archivo_corto}")
                    print(f"   -> Mode 'none':      {v_none}")
                    print(f"   -> Mode 'polar':     {v_polar}  {'⚠️ ¡FALLO!' if error_polar else '✅ OK'}")
                    print(f"   -> Mode 'cartesian': {v_cartesian}  {'⚠️ ¡FALLO!' if error_cartesian else '✅ OK'}")
                    print("-" * 60)
                    
            except Exception as e:
                print(f"💥 Error ao ler o arquivo {nombre_archivo_corto}: {e}")

    # 6. Evaluación final del contador de fallos
    print("-" * 60)
    print(f"Resumen: Revisados {total_archivos_revisados} archivos CSV en total.")
    
    if total_errores == 0:
        print("\n✅ TEST COMPLETADO CON ÉXITO")
        print("Todos los modos devuelven exactamente el mismo número de vecinos para las nubes analizadas: \n.")
        for dataset in clouds:
            print(f'- {dataset}\n')
    else:
        print(f"\n🚨 ERRORES: Se encontraron {total_errores} archivos con discrepancias de vecinos.")

In [9]:
verificar_integridad_vecinos(SUBSET_DATA, ALL_CLOUDS)

🚀 Iniciando test de validación de resultados...
------------------------------------------------------------
------------------------------------------------------------
Resumen: Revisados 90 archivos CSV en total.

✅ TEST COMPLETADO CON ÉXITO
Todos los modos devuelven exactamente el mismo número de vecinos para las nubes analizadas: 
.
- bildstein_station1_xyz_intensity_rgb

- sg27_station8_intensity_rgb

- 5145_54340

- Lille_0

- Paris_Luxembourg_6

- PNOA_2024_PNR_489-4672_NPC01

